# 과제

- 과제 제출
  - ETL 타겟 사이트 
    - https://www.starbucks.co.kr/store/store_map.do
  - 수집 내용
    - 서울시 스타벅스 매장 정보 추출 (Extract, Transform)
      - 매장명
      - 주소
      - 매장타입(general, reserve, generalDT)
  - 저장 방식
    - csv
  - 제출 정보
    - 기한 : 2026-03-16일 24시까지
    - 형식 : 이름.ipynb
    - 이메일 : edu2.ucoc@gmail.com

# 코드 작성

In [ ]:
from selenium import webdriver as wd
from selenium.webdriver.common.by import By
import pandas as pd
import time

# 객체 생성 및 타겟 사이트 진입
driver = wd.Chrome()  # 크롬 드라이버 객체 생성
driver.get("https://www.starbucks.co.kr/store/store_map.do")  # 타겟 사이트 진입

# 화면 로딩 될 때까지 기다리기
time.sleep(20) # 이용자가 몰리면 받아오는 속도가 매우 느려져서 시간을 늘림 ( 리뷰 시간에 돌려봄 )

# 서울시 스타벅스 매장 정보
# 수집 정보 : 매장명, 주소, 매장 타입(general, reserve, generalDT)

# 지역의 시/도 리스트 가져오기
sidos = driver.find_elements(By.CSS_SELECTOR, '.sido_arae_box li a')
sido_values = [[sido.get_attribute('textContent'),sido.get_attribute('data-sidocd')] for sido in sidos] # 시도명, 시도 코드

# 각 시/도 별로 반복 돌리기
for sido_name, sido_code in sido_values:
    driver.find_element(By.CSS_SELECTOR, '.store_map_layer_cont > .loca_search').click()    # 지역 검색 버튼 눌러서 시/도 띄우기 
    time.sleep(2)

    driver.find_element(By.CSS_SELECTOR, f'.sido_arae_box li a[data-sidocd="{sido_code}"]').click()    # 시/도를 선택하여 구/군 리스트 띄우기 ( 세종은 바로 결과 나옴 )
    time.sleep(3)

    if sido_code!=sido_values[-1][1]:   # 마지막 값인 세종은 구/군 선택이 없어서 제외
        # driver.find_element(By.CSS_SELECTOR, f'.gugun_arae_box li a[data-guguncd]').click()    # data-guguncd 는 전체 매장 리스트 보기. data-guguncd='코드' 로 구/군 선택 가능
        driver.find_elements(By.CSS_SELECTOR, f'.gugun_arae_box li a')[0].click()    # 리스트 첫번째가 전체 선택 버튼
    time.sleep(20)  # 이용자가 몰리면 받아오는 속도가 매우 느려져서 시간을 늘림 ( 리뷰 시간에 돌려봄 )
    
    # 지역의 모든 매장 리스트
    stores = driver.find_elements(By.CSS_SELECTOR, '.quickSearchResultBoxSidoGugun > li')

    # 각 매장의 정보를 저장
    store_values = [
        {
            "매장명" : store.get_attribute('data-name').strip(),
            "주소" : store.find_element(By.CLASS_NAME,'result_details').get_attribute('textContent').replace(',','|').strip()[:-9],  # 전화번호(9자리) 제외, (,) 제외
            "매장타입" : store.find_element(By.CSS_SELECTOR,'i').get_attribute('class').split('_')[1].strip()
        }
            for store in stores
    ]
    
    # pandas를 사용하여 csv로 저장
    df = pd.DataFrame(store_values)
    df.to_csv('박주용.csv')
    # df.to_csv(f'스타벅스_{sido_name}_매장.csv')   # 파일명 : 스타벅스_(지역이름)_매장.csv 으로 저장
    
    break; # 서울만 확인하기

driver.quit()